# OSASUD: быстрый просмотр результатов и обрезанные записи

**Этот ноутбук маленький: больших встроенных ячеек с данными нет.**
Запускайте сверху вниз. По умолчанию загружаются только код, проверенные таблицы и готовые рисунки (~2 МБ).
Признаки, обучение, bootstrap и графики заново НЕ пересчитываются. Результаты получены ранее реальным запуском на обрезанной выборке всех 30 пациентов.

Исходные записи: по 30 минут на пациента, 15 часов. При `LOAD_RAW=True` автоматически загружается отдельный архив 10,5 МБ. Большой Pickle 3,28 ГБ нигде не читается.
Для полного повторного расчёта: `RECOMPUTE_FEATURES=True`, затем `RECOMPUTE=True` в ячейке оценки. Это медленнее и для просмотра не требуется.

Сохранены patient-group nested CV, внутренний выбор порога, калибровка, 95% CI, flow-only сравнение, QC, проверка пульса, коэффициенты и подписанные 30-минутные рисунки. Пакет скачивается с GitHub; нужна сеть. Первый этап пишет прогресс.


In [ ]:
# @title Быстрый запуск: готовые результаты, без обучения и bootstrap
from pathlib import Path
import urllib.request, hashlib, zipfile, json, sys, time
import numpy as np
import pandas as pd
from IPython.display import display, Image
work=Path.cwd()/'OSASUD_FAST_work'
work.mkdir(exist_ok=True)
BASE='https://raw.githubusercontent.com/maksimilli/bitrix/osasud-30min-20261030/research/osasud-30min/'
def obtain(name, expected):
    path=work/name
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest()==expected:
        print('Используется уже загруженный файл:',name)
        return path
    print('Загрузка:',name,flush=True)
    last=[-1]
    def progress(block,size,total):
        if total>0:
            percent=min(100,int(block*size*100/total))
            if percent//25!=last[0]:
                last[0]=percent//25
                print(f'  {percent}%',flush=True)
    urllib.request.urlretrieve(BASE+name,str(path),reporthook=progress)
    assert hashlib.sha256(path.read_bytes()).hexdigest()==expected,'Неполная загрузка '+name
    return path
def unpack(path, names=None):
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if names is not None and name not in names:continue
            destination=(work/name).resolve()
            if not destination.is_relative_to(work.resolve()):raise ValueError('Неверный путь в ZIP')
            destination.parent.mkdir(exist_ok=True,parents=True)
            destination.write_bytes(archive.read(name))
print('Этап 1/2: открываем небольшие таблицы, код и готовые рисунки.',flush=True)
unpack(obtain('analysis_support.zip','90c4a6319d626d358b38fd9808969ef3db77f58a3925f85a41e0308a97b2da43'))
for name,digest in json.loads((work/'checksums.json').read_text()).items():
    assert hashlib.sha256((work/name).read_bytes()).hexdigest()==digest,name
manifest=json.loads((work/'manifest.json').read_text())
def load_raw():
    expected={f'patient_{pid}.npz':m['sha256'] for pid,m in manifest['examples'].items()}
    if not all((work/name).exists() and hashlib.sha256((work/name).read_bytes()).hexdigest()==sha for name,sha in expected.items()):
        unpack(obtain('OSASUD_30min_raw.zip','e34ddb9b9a811c0c29a65ab66ecc1269f19e56f1312e1223c53ee6fb89023408'),set(expected))
    for name,sha in expected.items():assert hashlib.sha256((work/name).read_bytes()).hexdigest()==sha,name
    print('Доступны исходные 30-минутные файлы всех 30 пациентов.')
LOAD_RAW=False  # True: дополнительно загрузить 10,5 МБ настоящих исходных сигналов
RECOMPUTE_FEATURES=False  # True: пересчитать признаки из маленьких исходников
sys.path.insert(0,str(work))
for name in ['signal_features','raw_windows','evaluate','plots']:sys.modules.pop(name,None)
if LOAD_RAW or RECOMPUTE_FEATURES:load_raw()
if RECOMPUTE_FEATURES:
    from raw_windows import rebuild
    frame,quality=rebuild(work)
else:
    frame=pd.read_csv(work/'features_all.csv',dtype={'patient':str})
    quality=pd.read_csv(work/'quality_all.csv',dtype={'patient':str})
print('Этап 2/2: готово. Далее открываются сохранённые результаты реального расчёта.',flush=True)
print('Пациентов:',quality.patient.nunique(),'окон до QC:',len(quality),'для оценки:',int(frame.y.notna().sum()))
display(frame.head(3))


## 1. Постановка, отбор и контроль качества
Одна строка оригинала — одна секунда; signal_pleth 80 Гц, PSG_Flow 20 Гц.
Анализируются по 30 минут 30 пациентов: всего 900 неперекрывающихся окон по 60 с до QC. Для каждого пациента выбран первый по времени 30-минутный блок с >=15 минутами, принятыми по QC. Если такого блока нет, берётся первый. Правило не использует event или успех модели. Такая выборка смещена по качеству и времени ночи и не представляет всю ночь.
Окна не пересекают разрывы времени. Окна с >=10 секунд APNEA — положительные,
с целой минутой NONE — отрицательные. Гипопноэ, неизвестные метки и 1–9 секунд
APNEA исключены из бинарной задачи; они остаются на исходных графиках.
Это не AHI и не детектор точных границ эпизодов.

Общая для моделей complete-case выборка: отвергаются окна с NaN/Inf в ФПГ или потоке,
постоянным участком ФПГ >=2 с, плоским ФПГ, недостаточными пиками или >20% неправдоподобных
PPI. Критерии заданы заранее, без использования event. Отдельно публикуются причины,
покрытие и вариация по пациентам. Исключение плохих окон может создавать selection bias;
качество на отвергнутых данных неизвестно. Эти простые QC-признаки не гарантируют отсутствие артефактов.
HR(bpm) из ЭКГ не используется как вход или критерий отбора модели.


In [ ]:
display(quality.groupby(['quality','label_eligible']).size().rename('windows').reset_index())
display(pd.read_csv(work/'coverage_by_patient.csv').head(20))


## 2. Предобработка и признаки — без утечки
Удалена RMS-нормализация всей записи и вычисление общего baseline. Нет глобальной
интерполяции и передискретизации. На отдельной копии каждого окна применяются фиксированные
фильтры: signal_pleth 0,5–8 Гц, PSG_Flow 0,05–2 Гц, Баттерворт 3-го порядка, zero-phase.
Это офлайн-обработка: будущие отсчёты внутри окна используются.

При RECOMPUTE_FEATURES=True признаки пересчитываются из небольших NPZ; по умолчанию читается проверенная таблица признаков. Фиксированные локальные RMS/SD/спектральные признаки не оценивают параметры по выборке,
поэтому их можно кешировать до CV: каждое окно независимо от остальных, event не используется.
Импутация признаков, standardization, class weighting и выбор C обучаются только в train
каждого внутреннего/внешнего фолда. Измерения с пропусками не интерполируются.

PPG: PR=60/median(PPI), SD PPI, RMSSD PPI, prominence median/CV, SD ФПГ, спектральная энтропия.
Пики: расстояние >=28 отсчётов, prominence >=0,3 SD; допустимый PPI 0,35–2 с.
Flow: RMS, SD, P90−P10, SD производной, спектральная энтропия.
Tp, App, ST, DT, Smax и E(Tp) в этой версии **не рассчитываются и не заявляются**.
Ссылка на непроверенное воспроизведение сторонней нейросети удалена.


In [ ]:
import inspect
from signal_features import features
print(inspect.getsource(features))
print('PPG:',manifest['PPG_features'])
print('Flow:',manifest['flow_features'])


## 3. Вложенная проверка по пациентам, калибровка и порог
Сравниваются PPG, flow_only и PPG_plus_flow на одинаковых окнах и внешних фолдах.
Внешний GroupKFold(5): пациенты целиком отложены, каждый пациент в тесте один раз.
Это НЕ leave-one-patient-out; каждый внешний тест содержит группу пациентов.
Внутри train — GroupKFold(3). C∈{0.1,1,10} выбирается по внутреннему average precision.
По внутренним OOF-оценкам выбранной модели обучается sigmoid/Platt-калибратор без class weighting.
Порог максимизирует внутренний F1 после калибровки. Внешние event и оценки не участвуют
ни в выборе C/порога, ни в обучении калибратора. Порог может различаться между фолдами.
Выбор порога и калибровка используют одну внутреннюю OOF-выборку; честная итоговая оценка
обеспечивается независимым внешним фолдом.

ROC-AUC, PR-AUC (именно average precision, не трапецеидальная площадь), Brier,
precision/recall/F1 и balanced accuracy считаются по внешним OOF-вероятностям.
Есть показатели каждого пациента и macro-усреднение, чтобы длинные записи не скрывали различия.
95% CI: 500 bootstrap-выборок целых пациентов, seed=42. Это условные CI по готовым OOF-прогнозам:
модели в bootstrap не переобучаются; полная неопределённость выбора модели ими не охватывается.
Не заявляем статистическое превосходство только по этим CI.


In [ ]:
RECOMPUTE=False  # True: повторить вложенный CV и 500 bootstrap-выборок
from evaluate import evaluate
if RECOMPUTE_FEATURES: RECOMPUTE=True
if RECOMPUTE:
    evaluate(frame,manifest['PPG_features'],manifest['flow_features'],out=work,boots=500)
metrics=pd.read_csv(work/'metrics.csv')
ci=pd.read_csv(work/'cluster_bootstrap_CI.csv')
display(metrics)
display(ci)
display(pd.read_csv(work/'fold_parameters.csv'))
display(pd.read_csv(work/'macro_patient_metrics.csv'))


## 4. ROC, PR и внешняя калибровка
Калибровочная кривая построена только по внешним OOF-прогнозам: идеальная диагональ
показывает совпадение вероятности с частотой события. Brier указан в таблице.
Калибровочные интервалы по квантилям, до 10 групп. PR-baseline — доля положительных окон.


In [ ]:
from plots import plots
REMAKE_FIGURES=False  # True: перестроить рисунки из исходников
if REMAKE_FIGURES or RECOMPUTE:
    load_raw()
    plots(work)
display(Image(filename=str(work/'figures/ROC_PR_calibration.png')))
display(pd.read_csv(work/'metrics_by_fold.csv'))


## 5. Проверка пульса по доступному HR(bpm)
Сравниваем минутный PR из signal_pleth с медианным HR(bpm) монитора по ЭКГ,
включая все принятые по QC окна, даже если они не подходят бинарной задаче.
Показаны MAE, RMSE, bias и Bland–Altman. Это проверка согласия частот, **не ручная
валидация каждого пика**. Пульсовой дефицит, аритмии и ошибки монитора могут создавать расхождения.
LoA описательные: повторные окна одного пациента зависимы. Для окончательной валидации пиков
нужна ручная выборочная разметка или сопоставление с валидированными ECG R-peaks.


In [ ]:
display(json.loads((work/'pulse_validation.json').read_text()))
display(Image(filename=str(work/'figures/pulse_validation.png')))


## 6. Длинные исходные фрагменты и вероятности моделей
По 30 минут пациентов 3,5,6; выбор Для иллюстраций выбраны первые три ID с >=60 секундами APNEA среди уже зафиксированных обрезанных записей. Только выбор рисунков использует event; состав обучающей/тестовой выборки и пороги от этого не зависят.
Исходники без изменения. Синий — expert event APNEA, жёлтый — HYPOPNEA.
Вероятности для этих пациентов — только из внешних тестовых фолдов.
Пунктир — порог из внутреннего CV этого фолда, красный — положительное минутное окно;
серый — окно не вошло в общую оценку. На графиках время в минутах; увеличение пиков — в секундах.

**Постобработка:** закрытия разрывов нет (max_gap=0); короткие предсказания не удаляются.
Каждое положительное окно равно 60 с. Соседние окна могут визуально соприкасаться;
через исключённые окна интервалы не объединяются. Event-level метрики не выдаются:
подсветка минут не определяет точные начало/конец клинического события.


In [ ]:
for patient in ['3','5','6']:
    print('patient',patient,manifest['examples'][patient])
    display(Image(filename=str(work/f'figures/patient_{patient}_30min.png')))
    if (work/f'figures/patient_{patient}_peaks.png').exists():
        display(Image(filename=str(work/f'figures/patient_{patient}_peaks.png')))


## 7. Коэффициенты и направления эффектов
Коэффициенты публикуются для каждого внешнего фолда. standardized_weight — коэффициент
базовой логистической регрессии; calibrated_weight учитывает наклон Platt-калибратора.
Среднее и SD по фолдам помогают увидеть нестабильность направления. Это ассоциации
при прочих признаках модели, не физиологические причинные эффекты.


In [ ]:
coefs=pd.read_csv(work/'coefficients.csv')
display(coefs)
display(coefs.groupby(['model','feature'])[['standardized_weight','calibrated_weight']].agg(['mean','std']))


## 8. Ограничения для текста статьи
Это исследовательская классификация минут APNEA против чистого NONE на отобранных по QC окнах.
Гипопноэ исключено; AASM-скоринг, AHI и клиническая диагностика не реализованы.
PSG_Flow здесь — носовой поток/давление. Правило/модель по нему не эквивалентны AASM:
для стандартного скоринга апноэ обычно используют оронозальный термический датчик;
для гипопноэ нужны дополнительные критерии десатурации и/или араузала по выбранным правилам.
В наборе нет всех необходимых каналов/процедур для независимого полного перескоринга.

Врач исходно использовал PSG, включая поток. Это не утечка event в модель,
но flow-only получает физиологический канал, близкий к основе эталона; это другая
постановка, чем детекция только по ФПГ. Для обструктивного/центрального подтипа нужны усилия дыхания.
Нет гарантии, что добавление ФПГ улучшит flow-only: результаты сообщаются как получены.

Данные — пациенты инсультного отделения с коморбидностями. Нужны внешняя валидация,
анализ пропусков/артефактов, ручная проверка пиков и отдельная оценка всех исключённых случаев.
Для повторного выделения признаков используется raw_windows.py и встроенные patient_N.npz. Большой исходный Pickle НЕ требуется. Весь анализ и все приведённые метрики относятся только к выбранным 30 минутам на пациента, не к полным ночам. 30 пациентов и редкие события всё ещё дают широкие CI; требуется внешняя валидация.
